# MaxPool2d

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mitchell-Mirano/sorix/blob/develop/docs/learn/layers/09-MaxPool2d.ipynb)
[![Open in GitHub](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/Mitchell-Mirano/sorix/blob/develop/docs/learn/layers/09-MaxPool2d.ipynb)
[![Open in Docs](https://img.shields.io/badge/Open%20in-Docs-blue?logo=readthedocs)](http://127.0.0.1:8000/sorix/learn/layers/09-MaxPool2d)

The **MaxPool2d** layer executes a translation-invariant sub-sampling operation. It selectively propagates only prominent local spatial activations, halving computational dimensions down the network stream and severely preventing model overfitting.

## Mathematical definition

There are **no learnable parameters** associated with Pooling calculations.
Given input tensor $\mathbf{X} \in \mathbb{R}^{N \times C \times H \times W}$, for any independently evaluated filter block dimensioned $K_H \times K_W$, we retrieve an output scalar for grid coordinate $Y_{n, c, h_{out}, w_{out}}$.

### Forward Computation

$$
\mathbf{Y}_{n, c, h_{out}, w_{out}} = \max_{k_h=0}^{K_H-1} \; \max_{k_w=0}^{K_W-1} \; \mathbf{X}_{n, c, h_{out} \cdot s_h + k_h, w_{out} \cdot s_w + k_w}
$$

### Dimensions Constraint

When configured intuitively ($s_h=K_H, s_w=K_W$) to eliminate redundancy overlap, resulting boundaries scale universally:
$$
H_{out} = \left\lfloor \frac{H - K_H}{s_h} + 1 \right\rfloor \quad \text{and} \quad W_{out} = \left\lfloor \frac{W - K_W}{s_w} + 1 \right\rfloor
$$

## Gradient Dispersal (Backpropagation)

While the layer does not update parameters, it is critical in routing analytic derivatives dynamically inside the Autograd system. If $L$ traces the global error scalar and $\frac{\partial \mathcal{L}}{\partial \mathbf{Y}}$ flows universally backwards, the pooling engine establishes selective sparsity criteria.

Only the individual local source coordinates $p^* = (h^*, w^*)$ historically yielding maximal values retain activation linkages. Ergo, all other spatial items nullify local inputs to exactly $0$.

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{X}_{n, c, h_{in}, w_{in}}} =
\begin{cases} 
      \left( \frac{\partial \mathcal{L}}{\partial \mathbf{Y}} \right)_{n, c, h_{out}, w_{out}} & \text{if } \mathbf{X}_{n, c, h_{in}, w_{in}} = \max(\dots) \text{ in the local mapped region} \\
      0 & \text{otherwise}
\end{cases}
$$

Computationally, this demands evaluating `argmax` matrices per input spatial block during forward execution, effectively saving indices bounds to distribute error matrices efficiently.

In [1]:
# Uncomment the next line and run this cell to install sorix
#!pip install 'sorix @ git+https://github.com/Mitchell-Mirano/sorix.git@develop'

In [2]:
from sorix import tensor
from sorix.nn import MaxPool2d
import numpy as np

In [3]:
# Create dense feature maps imitating intermediate network outputs
N, C, H, W = 1, 16, 28, 28
X = tensor(np.random.randn(N, C, H, W).astype(np.float32))
print("Input feature map shape:", X.shape)

Input feature map shape: sorix.Size([1, 16, 28, 28])


In [4]:
pool = MaxPool2d(kernel_size=2, stride=2)
Y = pool(X)

print("Output dimension cut in half:", Y.shape)

Output dimension cut in half: sorix.Size([1, 16, 14, 14])
